In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import locale
locale.getpreferredencoding()

'UTF-8'

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.0/793.0 kB 9.6 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-m

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import glob

from ultralytics import YOLO

# INFERENCE_IMAGES_DIRECTORY = '/content/gdrive/MyDrive/Disertatie/datasets/2000_images/valid/images/'
# INFERENCE_LABELS_DIRECTORY = '/content/gdrive/MyDrive/Disertatie/datasets/2000_images/valid/labels/'
INFERENCE_IMAGES_DIRECTORY = '/content/gdrive/MyDrive/Disertatie/datasets/5500_images/valid/images/'
INFERENCE_LABELS_DIRECTORY = '/content/gdrive/MyDrive/Disertatie/datasets/5500_images/valid/labels/'

In [ ]:
inference_image_list = []
for filename in os.listdir(INFERENCE_IMAGES_DIRECTORY):
  inference_image_list.append(filename)

In [ ]:
inference_label_list = []
for filename in os.listdir(INFERENCE_LABELS_DIRECTORY):
  inference_label_list.append(filename)

In [ ]:
def draw_image_labels(SAVE_DIRECTORY, IMAGE_DIRECTORY, LABEL_DIRECTORY, image, label):
    img = cv2.imread(os.path.join(IMAGE_DIRECTORY, image), cv2.IMREAD_COLOR)
    dh, dw, _ = img.shape

    fl = open(os.path.join(LABEL_DIRECTORY, label), 'r')
    data = fl.readlines()
    fl.close()

    for dt in data:

        id, x, y, w, h = map(float, dt.split(' '))

        l = int((x - w / 2) * dw)
        r = int((x + w / 2) * dw)
        t = int((y - h / 2) * dh)
        b = int((y + h / 2) * dh)

        if l < 0:
            l = 0
        if r > dw - 1:
            r = dw - 1
        if t < 0:
            t = 0
        if b > dh - 1:
            b = dh - 1

        if id == 1:
            cv2.rectangle(img, (l, t), (r, b), (0, 0, 255), 1) # rosu -> red - fire
        else:
            cv2.rectangle(img, (l, t), (r, b), (255, 0, 0), 1) # albastru -> blue - smoke

    cv2.imwrite(SAVE_DIRECTORY + image, img)

## 1000 images models

### SGD Models

#### 50 epochs model



In [ ]:
# Load a model
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/SGDmodels/sgd_50_epochs/ep50_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
# test_directory = '/content/gdrive/MyDrive/Disertatie/'
# test_image_list = ['PublicDataset01015.jpg', 'PublicDataset01019.jpg']

save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/50_epochs/norm_inference/'
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


0: 544x640 (no detections), 823.5ms
Speed: 23.2ms preprocess, 823.5ms inference, 15.0ms postprocess per image at shape (1, 3, 544, 640)

0: 384x640 1 fire, 349.4ms
Speed: 3.6ms preprocess, 349.4ms inference, 13.6ms postprocess per image at shape (1, 3, 384, 640)

0: 320x640 1 fire, 296.0ms
Speed: 3.6ms preprocess, 296.0ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 fire, 265.9ms
Speed: 5.2ms preprocess, 265.9ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 fires, 286.8ms
Speed: 3.7ms preprocess, 286.8ms inference, 3.4ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 fire, 562.9ms
Speed: 7.3ms preprocess, 562.9ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)

0: 384x640 2 smokes, 3 fires, 407.2ms
Speed: 3.9ms preprocess, 407.2ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 480x640 2 fires, 479.3ms
Speed: 3.4ms preprocess, 479.3ms inference, 1.0ms postprocess per

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/50_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/50_epochs/labeled_inference/'
for _image in inference_image_list:
    label_path = _image.strip().split('.')[0] + '.txt'
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

#### 100 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/SGDmodels/sgd_100_epochs/ep100_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/100_epochs/norm_inference/'
 # /content/gdrive/MyDrive/Disertatie/inference/1000_images_models/SGD_models/100_epochs/norm_inference
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


0: 544x640 2 fires, 895.5ms
Speed: 5.6ms preprocess, 895.5ms inference, 3.4ms postprocess per image at shape (1, 3, 544, 640)

0: 384x640 1 fire, 351.7ms
Speed: 4.9ms preprocess, 351.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 320x640 1 smoke, 2 fires, 282.5ms
Speed: 4.4ms preprocess, 282.5ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 fires, 269.7ms
Speed: 3.4ms preprocess, 269.7ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 fires, 307.6ms
Speed: 3.5ms preprocess, 307.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 fire, 278.0ms
Speed: 3.5ms preprocess, 278.0ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

0: 384x640 2 smokes, 3 fires, 330.6ms
Speed: 3.2ms preprocess, 330.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 480x640 12 smokes, 5 fires, 425.7ms
Speed: 4.5ms preprocess, 425.7ms inference, 1.2ms post

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/100_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/100_epochs/labeled_inference/'
for _image in inference_image_list:
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

#### 150 model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/SGDmodels/ep150_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/150_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
  # for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'Image: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)

Image: 1 

0: 544x640 1 fire, 810.4ms
Speed: 5.9ms preprocess, 810.4ms inference, 1.6ms postprocess per image at shape (1, 3, 544, 640)
Image: 2 

0: 384x640 1 fire, 539.8ms
Speed: 2.5ms preprocess, 539.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Image: 3 

0: 320x640 1 fire, 487.1ms
Speed: 4.9ms preprocess, 487.1ms inference, 1.3ms postprocess per image at shape (1, 3, 320, 640)
Image: 4 

0: 320x640 1 smoke, 2 fires, 396.6ms
Speed: 4.5ms preprocess, 396.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)
Image: 5 

0: 320x640 2 fires, 287.6ms
Speed: 3.2ms preprocess, 287.6ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)
Image: 6 

0: 320x640 1 fire, 273.9ms
Speed: 4.3ms preprocess, 273.9ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)
Image: 7 

0: 384x640 1 smoke, 3 fires, 331.6ms
Speed: 4.1ms preprocess, 331.6ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)
Image: 8 

0: 480x640 5 smo

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/150_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/SGD_models/150_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

### Adam Models

#### 50 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/AdamModels/ep50_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/50_epochs/norm_inference/'
# /content/gdrive/MyDrive/Disertatie/inference/1000_images_models/Adam_models/50_epochs/norm_inference
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\n Image: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


 Image: 1 

0: 544x640 (no detections), 528.6ms
Speed: 4.1ms preprocess, 528.6ms inference, 0.5ms postprocess per image at shape (1, 3, 544, 640)

 Image: 2 

0: 384x640 (no detections), 372.3ms
Speed: 6.4ms preprocess, 372.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

 Image: 3 

0: 320x640 1 smoke, 2 fires, 268.2ms
Speed: 6.1ms preprocess, 268.2ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

 Image: 4 

0: 320x640 1 smoke, 3 fires, 296.1ms
Speed: 3.1ms preprocess, 296.1ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 640)

 Image: 5 

0: 320x640 2 fires, 265.3ms
Speed: 3.0ms preprocess, 265.3ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

 Image: 6 

0: 320x640 1 fire, 268.5ms
Speed: 3.2ms preprocess, 268.5ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

 Image: 7 

0: 384x640 1 smoke, 3 fires, 340.1ms
Speed: 4.4ms preprocess, 340.1ms inference, 0.8ms postprocess per image at shape (1

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/50_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/50_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} \n ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
 
Image: 2 
 
Image: 3 
 
Image: 4 
 
Image: 5 
 
Image: 6 
 
Image: 7 
 
Image: 8 
 
Image: 9 
 
Image: 10 
 
Image: 11 
 
Image: 12 
 
Image: 13 
 
Image: 14 
 
Image: 15 
 
Image: 16 
 
Image: 17 
 
Image: 18 
 
Image: 19 
 
Image: 20 
 
Image: 21 
 
Image: 22 
 
Image: 23 
 
Image: 24 
 
Image: 25 
 
Image: 26 
 
Image: 27 
 
Image: 28 
 
Image: 29 
 
Image: 30 
 
Image: 31 
 
Image: 32 
 
Image: 33 
 
Image: 34 
 
Image: 35 
 
Image: 36 
 
Image: 37 
 
Image: 38 
 
Image: 39 
 
Image: 40 
 
Image: 41 
 
Image: 42 
 
Image: 43 
 
Image: 44 
 
Image: 45 
 
Image: 46 
 
Image: 47 
 
Image: 48 
 
Image: 49 
 
Image: 50 
 
Image: 51 
 
Image: 52 
 
Image: 53 
 
Image: 54 
 
Image: 55 
 
Image: 56 
 
Image: 57 
 
Image: 58 
 
Image: 59 
 
Image: 60 
 
Image: 61 
 
Image: 62 
 
Image: 63 
 
Image: 64 
 
Image: 65 
 
Image: 66 
 
Image: 67 
 
Image: 68 
 
Image: 69 
 
Image: 70 
 
Image: 71 
 
Image: 72 
 
Image: 73 
 
Image: 74 
 
Image: 75 
 
Image: 76 
 
Image: 77 
 
Image: 7

#### 100 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/AdamModels/ep100_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model
# /content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/AdamModels/ep100_Adam_01lr_b16/weights/best.pt

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/100_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 2 fires, 566.8ms
Speed: 7.0ms preprocess, 566.8ms inference, 1.3ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 358.0ms
Speed: 5.2ms preprocess, 358.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 fire, 279.4ms
Speed: 3.0ms preprocess, 279.4ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 2 fires, 333.1ms
Speed: 4.2ms preprocess, 333.1ms inference, 1.3ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 502.6ms
Speed: 3.3ms preprocess, 502.6ms inference, 1.3ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 431.2ms
Speed: 3.4ms preprocess, 431.2ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 558.0ms
Speed: 4.3ms preprocess, 558.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

0: 480x640 2 smo

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/100_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/100_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'\nImage: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)


Image: 1 

Image: 2 

Image: 3 

Image: 4 

Image: 5 

Image: 6 

Image: 7 

Image: 8 

Image: 9 

Image: 10 

Image: 11 

Image: 12 

Image: 13 

Image: 14 

Image: 15 

Image: 16 

Image: 17 

Image: 18 

Image: 19 

Image: 20 

Image: 21 

Image: 22 

Image: 23 

Image: 24 

Image: 25 

Image: 26 

Image: 27 

Image: 28 

Image: 29 

Image: 30 

Image: 31 

Image: 32 

Image: 33 

Image: 34 

Image: 35 

Image: 36 

Image: 37 

Image: 38 

Image: 39 

Image: 40 

Image: 41 

Image: 42 

Image: 43 

Image: 44 

Image: 45 

Image: 46 

Image: 47 

Image: 48 

Image: 49 

Image: 50 

Image: 51 

Image: 52 

Image: 53 

Image: 54 

Image: 55 

Image: 56 

Image: 57 

Image: 58 

Image: 59 

Image: 60 

Image: 61 

Image: 62 

Image: 63 

Image: 64 

Image: 65 

Image: 66 

Image: 67 

Image: 68 

Image: 69 

Image: 70 

Image: 71 

Image: 72 

Image: 73 

Image: 74 

Image: 75 

Image: 76 

Image: 77 

Image: 78 

Image: 79 

Image: 80 

Image: 81 

Image: 82 

Image: 83 

Image: 84 



#### 150 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/AdamModels/ep150_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model
# /content/gdrive/MyDrive/Disertatie/Modele/yolov8s/1000imagesModels/AdamModels/ep150_Adam_01lr_b16/weights/best.pt

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/150_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 1 fire, 539.4ms
Speed: 4.6ms preprocess, 539.4ms inference, 1.0ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 354.3ms
Speed: 5.7ms preprocess, 354.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 fire, 310.8ms
Speed: 3.7ms preprocess, 310.8ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 2 fires, 280.5ms
Speed: 5.0ms preprocess, 280.5ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 284.1ms
Speed: 6.0ms preprocess, 284.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 294.4ms
Speed: 4.4ms preprocess, 294.4ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 335.5ms
Speed: 3.1ms preprocess, 335.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

0: 480x6

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/150_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/1000_images_models/Adam_models/150_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'\nImage: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)


Image: 1 

Image: 2 

Image: 3 

Image: 4 

Image: 5 

Image: 6 

Image: 7 

Image: 8 

Image: 9 

Image: 10 

Image: 11 

Image: 12 

Image: 13 

Image: 14 

Image: 15 

Image: 16 

Image: 17 

Image: 18 

Image: 19 

Image: 20 

Image: 21 

Image: 22 

Image: 23 

Image: 24 

Image: 25 

Image: 26 

Image: 27 

Image: 28 

Image: 29 

Image: 30 

Image: 31 

Image: 32 

Image: 33 

Image: 34 

Image: 35 

Image: 36 

Image: 37 

Image: 38 

Image: 39 

Image: 40 

Image: 41 

Image: 42 

Image: 43 

Image: 44 

Image: 45 

Image: 46 

Image: 47 

Image: 48 

Image: 49 

Image: 50 

Image: 51 

Image: 52 

Image: 53 

Image: 54 

Image: 55 

Image: 56 

Image: 57 

Image: 58 

Image: 59 

Image: 60 

Image: 61 

Image: 62 

Image: 63 

Image: 64 

Image: 65 

Image: 66 

Image: 67 

Image: 68 

Image: 69 

Image: 70 

Image: 71 

Image: 72 

Image: 73 

Image: 74 

Image: 75 

Image: 76 

Image: 77 

Image: 78 

Image: 79 

Image: 80 

Image: 81 

Image: 82 

Image: 83 

Image: 84 



## 2000 images models

### SGD Models

#### 50 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/2000imagesModels/SGDmodels/ep50_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/50_epochs/norm_inference/'
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


0: 544x640 (no detections), 840.9ms
Speed: 5.1ms preprocess, 840.9ms inference, 0.7ms postprocess per image at shape (1, 3, 544, 640)

0: 384x640 (no detections), 376.5ms
Speed: 6.0ms preprocess, 376.5ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 320x640 1 smoke, 3 fires, 290.7ms
Speed: 4.8ms preprocess, 290.7ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 smoke, 3 fires, 276.8ms
Speed: 3.0ms preprocess, 276.8ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 fires, 275.8ms
Speed: 4.7ms preprocess, 275.8ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 2 fires, 276.5ms
Speed: 4.8ms preprocess, 276.5ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

0: 384x640 1 smoke, 3 fires, 338.5ms
Speed: 4.1ms preprocess, 338.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 480x640 1 fire, 417.5ms
Speed: 4.0ms preprocess, 417.5ms inferen

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/50_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/50_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 100 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/2000imagesModels/SGDmodels/ep100_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/100_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'Image: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)

Image: 1 

0: 544x640 1 fire, 555.3ms
Speed: 9.5ms preprocess, 555.3ms inference, 0.9ms postprocess per image at shape (1, 3, 544, 640)
Image: 2 

0: 384x640 1 fire, 370.8ms
Speed: 4.3ms preprocess, 370.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)
Image: 3 

0: 320x640 1 smoke, 3 fires, 311.8ms
Speed: 3.2ms preprocess, 311.8ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)
Image: 4 

0: 320x640 1 smoke, 3 fires, 302.9ms
Speed: 3.9ms preprocess, 302.9ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)
Image: 5 

0: 320x640 2 fires, 289.7ms
Speed: 6.4ms preprocess, 289.7ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)
Image: 6 

0: 320x640 2 fires, 328.5ms
Speed: 3.8ms preprocess, 328.5ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)
Image: 7 

0: 384x640 1 smoke, 3 fires, 337.8ms
Speed: 6.0ms preprocess, 337.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Image: 8 

0: 48

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/100_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/100_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 150 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/2000imagesModels/SGDmodels/ep150_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/150_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 (no detections), 615.9ms
Speed: 5.7ms preprocess, 615.9ms inference, 0.5ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 (no detections), 357.1ms
Speed: 3.3ms preprocess, 357.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 fire, 280.6ms
Speed: 3.7ms preprocess, 280.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 2 fires, 279.5ms
Speed: 3.2ms preprocess, 279.5ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 294.0ms
Speed: 3.4ms preprocess, 294.0ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 291.0ms
Speed: 3.8ms preprocess, 291.0ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 348.0ms
Speed: 3.7ms preprocess, 348.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

I

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/150_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/SGD_models/150_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

### Adam Models

#### 50 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/2000imagesModels/AdamModels/ep50_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/50_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 (no detections), 504.6ms
Speed: 6.3ms preprocess, 504.6ms inference, 0.6ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 (no detections), 342.5ms
Speed: 6.1ms preprocess, 342.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 smoke, 3 fires, 279.6ms
Speed: 3.5ms preprocess, 279.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 1 fire, 271.4ms
Speed: 4.3ms preprocess, 271.4ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 287.7ms
Speed: 7.6ms preprocess, 287.7ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 2 fires, 302.2ms
Speed: 3.3ms preprocess, 302.2ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 2 fires, 343.8ms
Speed: 3.1ms preprocess, 343.8ms inference, 0.8ms postprocess per image at shape (1, 3, 38

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/50_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/50_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 100 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/2000imagesModels/AdamModels/ep100_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/100_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 (no detections), 499.3ms
Speed: 6.6ms preprocess, 499.3ms inference, 0.8ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 (no detections), 333.1ms
Speed: 3.9ms preprocess, 333.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 smoke, 3 fires, 287.3ms
Speed: 4.3ms preprocess, 287.3ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 fire, 285.1ms
Speed: 6.2ms preprocess, 285.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 280.3ms
Speed: 4.9ms preprocess, 280.3ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 2 fires, 323.7ms
Speed: 3.0ms preprocess, 323.7ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 2 fires, 325.3ms
Speed: 5.0ms preprocess, 325.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)



In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/100_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/100_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 150 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/2000imagesModels/AdamModels/ep150_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/150_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 1 fire, 447.2ms
Speed: 5.1ms preprocess, 447.2ms inference, 0.8ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 352.9ms
Speed: 4.9ms preprocess, 352.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 smoke, 3 fires, 290.9ms
Speed: 4.6ms preprocess, 290.9ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 2 fires, 280.1ms
Speed: 3.1ms preprocess, 280.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 290.5ms
Speed: 4.6ms preprocess, 290.5ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 4 fires, 288.1ms
Speed: 3.4ms preprocess, 288.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 331.5ms
Speed: 4.5ms preprocess, 331.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 8

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/150_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/2000_images_models/Adam_models/150_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

## 5500 images models

### SGD Models

#### 50 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/5500imagesModels/SGDmodels/ep50_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/50_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 (no detections), 520.9ms
Speed: 6.6ms preprocess, 520.9ms inference, 0.5ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 544.5ms
Speed: 2.5ms preprocess, 544.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 fire, 488.3ms
Speed: 3.5ms preprocess, 488.3ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 3 fires, 446.7ms
Speed: 3.1ms preprocess, 446.7ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 461.5ms
Speed: 3.3ms preprocess, 461.5ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 498.5ms
Speed: 3.3ms preprocess, 498.5ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 374.8ms
Speed: 3.2ms preprocess, 374.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

0: 480x6

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/50_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/50_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 100 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/5500imagesModels/SGDmodels/ep100_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/100_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 1 fire, 889.2ms
Speed: 4.7ms preprocess, 889.2ms inference, 1.5ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 548.0ms
Speed: 4.9ms preprocess, 548.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 2 fires, 480.7ms
Speed: 3.2ms preprocess, 480.7ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 3 fires, 459.9ms
Speed: 3.5ms preprocess, 459.9ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 531.2ms
Speed: 3.2ms preprocess, 531.2ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 333.0ms
Speed: 3.1ms preprocess, 333.0ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 343.0ms
Speed: 3.6ms preprocess, 343.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

0: 480x

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/100_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/100_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 150 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/5500imagesModels/SGDmodels/ep150_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/150_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 1 fire, 862.9ms
Speed: 6.8ms preprocess, 862.9ms inference, 1.2ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 532.2ms
Speed: 4.4ms preprocess, 532.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 fire, 457.7ms
Speed: 5.4ms preprocess, 457.7ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 2 fires, 466.3ms
Speed: 3.9ms preprocess, 466.3ms inference, 1.4ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 451.3ms
Speed: 3.9ms preprocess, 451.3ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 292.5ms
Speed: 3.9ms preprocess, 292.5ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 323.3ms
Speed: 3.5ms preprocess, 323.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

0: 480x6

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/150_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/SGD_models/150_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

### Adam Models

#### 50 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/5500imagesModels/AdamModels/ep50_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/50_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 3 fires, 554.6ms
Speed: 7.4ms preprocess, 554.6ms inference, 0.9ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 352.7ms
Speed: 4.9ms preprocess, 352.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 fire, 293.0ms
Speed: 4.8ms preprocess, 293.0ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 1 fire, 289.4ms
Speed: 4.9ms preprocess, 289.4ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 291.0ms
Speed: 4.0ms preprocess, 291.0ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 277.6ms
Speed: 3.7ms preprocess, 277.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 2 fires, 344.0ms
Speed: 3.2ms preprocess, 344.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

0: 480x6

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/50_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/50_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 100 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/5500imagesModels/AdamModels/ep100_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/100_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 1 fire, 545.2ms
Speed: 5.0ms preprocess, 545.2ms inference, 1.0ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 340.3ms
Speed: 5.8ms preprocess, 340.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 smoke, 2 fires, 276.9ms
Speed: 4.6ms preprocess, 276.9ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 2 fires, 305.9ms
Speed: 4.4ms preprocess, 305.9ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 275.5ms
Speed: 4.8ms preprocess, 275.5ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 278.9ms
Speed: 4.2ms preprocess, 278.9ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 2 smokes, 2 fires, 358.7ms
Speed: 4.9ms preprocess, 358.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

Image: 8

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/100_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/100_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

#### 150 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/5500imagesModels/AdamModels/ep150_Adam_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/150_epochs/norm_inference/'
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 1 fire, 551.5ms
Speed: 4.4ms preprocess, 551.5ms inference, 0.9ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 385.4ms
Speed: 5.7ms preprocess, 385.4ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 smoke, 2 fires, 378.3ms
Speed: 3.1ms preprocess, 378.3ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 2 fires, 277.1ms
Speed: 3.7ms preprocess, 277.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 304.0ms
Speed: 3.3ms preprocess, 304.0ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 265.8ms
Speed: 5.4ms preprocess, 265.8ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 342.8ms
Speed: 5.3ms preprocess, 342.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/150_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/Adam_models/150_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

### 300 epochs model with alternative dataset

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/ep300_Adam_01lr_b16_alt_dataset/weights/best.pt')
# /content/gdrive/MyDrive/Disertatie/Modele/yolov8s/ep300_Adam_01lr_b16_alt_dataset/weights/best.pt

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/300_epochs/norm_inference/'
# /content/gdrive/MyDrive/Disertatie/inference/5500_images_models/300_epochs/norm_inference/
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 (no detections), 543.3ms
Speed: 4.1ms preprocess, 543.3ms inference, 0.5ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 362.7ms
Speed: 5.7ms preprocess, 362.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 smoke, 3 fires, 294.6ms
Speed: 5.3ms preprocess, 294.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 2 fires, 284.1ms
Speed: 4.9ms preprocess, 284.1ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 1 smoke, 2 fires, 318.7ms
Speed: 5.2ms preprocess, 318.7ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 smoke, 1 fire, 447.6ms
Speed: 3.1ms preprocess, 447.6ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 2 fires, 548.8ms
Speed: 3.1ms preprocess, 548.8ms inference, 1.2ms postprocess per image at shape 

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/300_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/300_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

### 450 epochs model

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/ep450_Adam_01lr_b16/weights/best.pt')
# /content/gdrive/MyDrive/Disertatie/Modele/yolov8s/ep450_Adam_01lr_b16/weights/best.pt

In [ ]:
save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/450_epochs/norm_inference/'
# /content/gdrive/MyDrive/Disertatie/inference/5500_images_models/450_epochs/norm_inference
count = 1
for image_path in inference_image_list:
# for image_path in inference_image_list
  image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
  # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  print(f'\nImage: {count} ')
  count += 1
  results = model(image)[0]

  # Process results list
  for result in results.boxes.data.tolist():
      x1, y1, x2, y2, score, class_id = result
      if int(class_id) == 1:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
      else:
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
        cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

  image_name = image_path.strip().split('.')[0]
  cv2.imwrite(save_directory + image_name + '_result.jpg', image)


Image: 1 

0: 544x640 2 fires, 851.2ms
Speed: 5.4ms preprocess, 851.2ms inference, 1.2ms postprocess per image at shape (1, 3, 544, 640)

Image: 2 

0: 384x640 1 fire, 356.4ms
Speed: 5.2ms preprocess, 356.4ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Image: 3 

0: 320x640 1 fire, 286.8ms
Speed: 5.0ms preprocess, 286.8ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 4 

0: 320x640 1 smoke, 4 fires, 285.1ms
Speed: 3.4ms preprocess, 285.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 5 

0: 320x640 2 fires, 290.6ms
Speed: 3.2ms preprocess, 290.6ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 6 

0: 320x640 1 fire, 281.5ms
Speed: 3.2ms preprocess, 281.5ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

Image: 7 

0: 384x640 1 smoke, 3 fires, 332.9ms
Speed: 4.8ms preprocess, 332.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

Image: 8 

0: 480x

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/450_epochs/norm_inference/'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference_550/5500_images_models/450_epochs/labeled_inference/'
count = 1
for _image in inference_image_list:
    print(f'Image: {count} ')
    count += 1
    label_path = _image.strip().split('.')[0] + '.txt'
    # print(label_path)
    image_path = _image.strip().split('.')[0] + '_result.jpg'
    # print(image_path)
    draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

Image: 1 
Image: 2 
Image: 3 
Image: 4 
Image: 5 
Image: 6 
Image: 7 
Image: 8 
Image: 9 
Image: 10 
Image: 11 
Image: 12 
Image: 13 
Image: 14 
Image: 15 
Image: 16 
Image: 17 
Image: 18 
Image: 19 
Image: 20 
Image: 21 
Image: 22 
Image: 23 
Image: 24 
Image: 25 
Image: 26 
Image: 27 
Image: 28 
Image: 29 
Image: 30 
Image: 31 
Image: 32 
Image: 33 
Image: 34 
Image: 35 
Image: 36 
Image: 37 
Image: 38 
Image: 39 
Image: 40 
Image: 41 
Image: 42 
Image: 43 
Image: 44 
Image: 45 
Image: 46 
Image: 47 
Image: 48 
Image: 49 
Image: 50 
Image: 51 
Image: 52 
Image: 53 
Image: 54 
Image: 55 
Image: 56 
Image: 57 
Image: 58 
Image: 59 
Image: 60 
Image: 61 
Image: 62 
Image: 63 
Image: 64 
Image: 65 
Image: 66 
Image: 67 
Image: 68 
Image: 69 
Image: 70 
Image: 71 
Image: 72 
Image: 73 
Image: 74 
Image: 75 
Image: 76 
Image: 77 
Image: 78 
Image: 79 
Image: 80 
Image: 81 
Image: 82 
Image: 83 
Image: 84 
Image: 85 
Image: 86 
Image: 87 
Image: 88 
Image: 89 
Image: 90 
Image: 91 
Image: 9

## single image

In [ ]:
# Load a model
model = YOLO('/content/gdrive/MyDrive/Disertatie/Modele/yolov8s/5500imagesModels/SGDmodels/ep50_SGD_01lr_b16/weights/best.pt')  # pretrained YOLOv8s model

In [ ]:
INFERENCE_IMAGES_DIRECTORY = '/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/'
INFERENCE_LABELS_DIRECTORY = '/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/labels/'
save_directory = '/content/gdrive/MyDrive/Disertatie/inference/lucrare_scrisa'
image_path = 'WEB04062.jpg'

image = cv2.imread(INFERENCE_IMAGES_DIRECTORY + image_path, cv2.IMREAD_COLOR)
results = model(image)[0]

# Process results list
for result in results.boxes.data.tolist():
    x1, y1, x2, y2, score, class_id = result
    if int(class_id) == 1:
      cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 10), 2) # verde -> green - fire
      cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                          cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 10), 1, cv2.LINE_AA)
    else:
      cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 250, 200), 2) # yellow -> galben - smoke
      cv2.putText(image, results.names[int(class_id)].lower(), (int(x1 + 3), int(y1 + 15)),
                          cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 250, 200), 1, cv2.LINE_AA)

image_name = image_path.strip().split('.')[0]
cv2.imwrite(save_directory + image_name + '_result.jpg', image)


0: 448x640 1 smoke, 3 fires, 867.3ms
Speed: 6.1ms preprocess, 867.3ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)


True

In [ ]:
folder_directory = '/content/gdrive/MyDrive/Disertatie/inference/lucrare_scrisa'
labeled_save_directory = '/content/gdrive/MyDrive/Disertatie/inference/lucrare_scrisa'
label_path = 'WEB04062.txt'

draw_image_labels(labeled_save_directory, folder_directory, INFERENCE_LABELS_DIRECTORY, image_path, label_path)

AttributeError: 'NoneType' object has no attribute 'shape'